# 🥇 Gold Price Predictor — Indian Market (INR)
Fetches real gold price data, converts to INR, trains an LSTM model, and predicts the next MCX gold price.

- **Gold Ticker:** `GC=F` (Gold Futures, USD) converted to INR
- **FX Ticker:** `INR=X` (USD to INR live rate)
- **Output:** Price in ₹ per 10 grams (MCX standard)

In [1]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install yfinance pandas numpy scikit-learn tensorflow matplotlib --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
# ── 2. Imports ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print(f'TensorFlow: {tf.__version__}')
print('All libraries loaded ✅')

Matplotlib is building the font cache; this may take a moment.


TensorFlow: 2.21.0
All libraries loaded ✅


In [6]:
# ── 3. Fetch Gold + USD/INR Data ─────────────────────────────────────────────
PERIOD   = '5y'
INTERVAL = '1d'

# Gold Futures in USD (troy oz)
gold_raw = yf.download('GC=F', period=PERIOD, interval=INTERVAL, progress=False)
print(gold_raw.head(3))
# USD to INR exchange rate
fx_raw = yf.download('INR=X', period=PERIOD, interval=INTERVAL, progress=False)
print(fx_raw.head(3))
gold_raw.dropna(inplace=True)
fx_raw.dropna(inplace=True)

print(f'Gold data : {len(gold_raw)} days ({gold_raw.index[0].date()} → {gold_raw.index[-1].date()})')
print(f'FX data   : {len(fx_raw)} days')

# ── Conversion: USD/troy_oz → INR/10g ────────────────────────────────────────
# 1 troy oz = 31.1035 grams
# price per gram (INR) = price_usd_oz * usd_inr / 31.1035
# price per 10g (INR)  = above * 10

TROY_OZ_TO_GRAMS = 31.1035

# Align on common dates
fx_close = fx_raw[['Close']].rename(columns={'Close': 'USDINR'})
gold_close = gold_raw[['Open','High','Low','Close','Volume']]

df = gold_close.join(fx_close, how='inner')
df.dropna(inplace=True)
# Convert all OHLC prices to ₹ per 10g
for col in ['Open','High','Low','Close']:
    df[col] = (df[col] * df['USDINR'] / TROY_OZ_TO_GRAMS) * 10

print(f'\nCombined dataset: {len(df)} days')
print(f'Latest gold price: ₹{df["Close"].iloc[-1]:,.0f} per 10g')
df.tail(3)

Price             Close         High          Low         Open Volume
Ticker             GC=F         GC=F         GC=F         GC=F   GC=F
Date                                                                 
2021-04-14  1734.900024  1745.900024  1732.699951  1741.300049   1037
2021-04-15  1765.400024  1767.900024  1740.099976  1740.099976    368
2021-04-16  1779.000000  1779.500000  1770.099976  1770.400024    625
Price           Close       High        Low       Open Volume
Ticker          INR=X      INR=X      INR=X      INR=X  INR=X
Date                                                         
2021-04-14  75.181503  75.260002  75.003899  75.181900      0
2021-04-15  75.026100  75.362999  74.744003  75.054001      0
2021-04-16  74.727402  74.836502  74.297699  74.727402      0
Gold data : 1258 days (2021-04-14 → 2026-04-14)
FX data   : 1300 days


ValueError: Columns must be same length as key

In [ ]:
# ── 4. Feature Engineering ───────────────────────────────────────────────────
# Moving Averages
df['MA_10'] = df['Close'].rolling(10).mean()
df['MA_20'] = df['Close'].rolling(20).mean()
df['MA_50'] = df['Close'].rolling(50).mean()

# RSI (14-period)
delta = df['Close'].diff()
gain  = delta.clip(lower=0).rolling(14).mean()
loss  = (-delta.clip(upper=0)).rolling(14).mean()
df['RSI'] = 100 - (100 / (1 + gain / loss))

# Bollinger Bands
std20 = df['Close'].rolling(20).std()
df['BB_upper'] = df['MA_20'] + 2 * std20
df['BB_lower'] = df['MA_20'] - 2 * std20

# USD/INR as a feature (impacts gold INR price)
df['USDINR_MA5'] = df['USDINR'].rolling(5).mean()

# Daily Return & Volatility
df['Daily_Return'] = df['Close'].pct_change()
df['Volatility']   = df['Daily_Return'].rolling(10).std()

df.dropna(inplace=True)
print(f'Feature matrix: {df.shape}')
df.tail(3)

In [ ]:
# ── 5. Scale & Build Sequences ───────────────────────────────────────────────
FEATURES = ['Open','High','Low','Close','Volume','USDINR',
            'MA_10','MA_20','MA_50','RSI','BB_upper','BB_lower',
            'USDINR_MA5','Daily_Return','Volatility']
TARGET_COL = 'Close'
SEQ_LEN    = 60

scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[FEATURES])

close_idx = FEATURES.index(TARGET_COL)

X, y = [], []
for i in range(SEQ_LEN, len(scaled)):
    X.append(scaled[i - SEQ_LEN:i])
    y.append(scaled[i, close_idx])

X, y = np.array(X), np.array(y)

split   = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

In [ ]:
# ── 6. Build LSTM Model ──────────────────────────────────────────────────────
tf.random.set_seed(42)

model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(SEQ_LEN, len(FEATURES))),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

In [ ]:
# ── 7. Train ─────────────────────────────────────────────────────────────────
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs           = 100,
    batch_size       = 32,
    validation_split = 0.1,
    callbacks        = [early_stop],
    verbose          = 1
)

In [ ]:
# ── 8. Evaluate ──────────────────────────────────────────────────────────────
def inverse_close(scaled_vals):
    dummy = np.zeros((len(scaled_vals), len(FEATURES)))
    dummy[:, close_idx] = scaled_vals
    return scaler.inverse_transform(dummy)[:, close_idx]

y_pred_scaled = model.predict(X_test).flatten()
y_pred = inverse_close(y_pred_scaled)
y_true = inverse_close(y_test)

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f'MAE  : ₹{mae:,.0f}')
print(f'RMSE : ₹{rmse:,.0f}')
print(f'MAPE : {mape:.2f}%')

In [ ]:
# ── 9. Plot: Actual vs Predicted ─────────────────────────────────────────────
test_dates = df.index[SEQ_LEN + split:]

fig, axes = plt.subplots(2, 1, figsize=(14, 10), facecolor='#0d0d0d')

ax1 = axes[0]
ax1.set_facecolor('#0d0d0d')
ax1.plot(test_dates, y_true, color='#FFD700', linewidth=1.5, label='Actual (MCX INR)')
ax1.plot(test_dates, y_pred, color='#FF6B35', linewidth=1.5, linestyle='--', label='Predicted')
ax1.set_title('Gold Price (₹/10g) — Actual vs Predicted', color='#FFD700', fontsize=14, pad=10)
ax1.set_ylabel('Price (₹ per 10g)', color='white')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
ax1.tick_params(colors='white')
ax1.legend(facecolor='#1a1a1a', labelcolor='white')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.grid(alpha=0.15, color='white')
for spine in ax1.spines.values(): spine.set_color('#333')

ax2 = axes[1]
ax2.set_facecolor('#0d0d0d')
ax2.plot(history.history['loss'],     color='#FFD700', label='Train Loss')
ax2.plot(history.history['val_loss'], color='#FF6B35', label='Val Loss')
ax2.set_title('Training & Validation Loss', color='#FFD700', fontsize=14, pad=10)
ax2.set_xlabel('Epoch', color='white')
ax2.set_ylabel('MSE Loss', color='white')
ax2.tick_params(colors='white')
ax2.legend(facecolor='#1a1a1a', labelcolor='white')
ax2.grid(alpha=0.15, color='white')
for spine in ax2.spines.values(): spine.set_color('#333')

plt.tight_layout()
plt.savefig('gold_prediction_india.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('Chart saved → gold_prediction_india.png')

In [ ]:
# ── 10. Predict Next Trading Day Price ───────────────────────────────────────
last_seq    = scaled[-SEQ_LEN:].reshape(1, SEQ_LEN, len(FEATURES))
next_scaled = model.predict(last_seq).flatten()
next_price  = inverse_close(next_scaled)[0]

last_price = df['Close'].iloc[-1]
change     = next_price - last_price
direction  = '📈 UP' if change >= 0 else '📉 DOWN'
last_fx    = df['USDINR'].iloc[-1]

print('=' * 50)
print(f'  USD/INR rate     : ₹{last_fx:.2f}')
print(f'  Last price (MCX) : ₹{last_price:,.0f} / 10g')
print(f'  Predicted next   : ₹{next_price:,.0f} / 10g')
print(f'  Expected change  : {direction}  ₹{change:+,.0f}')
print('=' * 50)

In [ ]:
# ── 11. Save model & scaler ───────────────────────────────────────────────────
import joblib

model.save('gold_lstm_india.keras')
joblib.dump(scaler, 'gold_scaler_india.pkl')

print('Model saved  → gold_lstm_india.keras')
print('Scaler saved → gold_scaler_india.pkl')